Cell 1 – Install / Import

In [3]:
# ============================================================
# 📦 Telco Churn Prediction — XGBoost
# Diamond • Ultimate • Elite • Netflix‑Ready
# ============================================================

# === Step 1: Imports ===
import pandas as pd
import numpy as np
import joblib
import os

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from datetime import datetime

# === Step 2: Config ===
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = "../backend/models/telco/"
MODEL_NAME = "xgb_model.pkl"
FEATURES_NAME = "feature_names_xgb.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# === Step 3: Load Data ===
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset with shape: {df.shape}")

# === Step 4: Clean & Preprocess ===

# Handle whitespace / missing TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Convert target to binary
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop identifier
df = df.drop(columns=["customerID"])

# Separate features & label
y = df["Churn"]
X = df.drop(columns=["Churn"])

# One‑hot encode categoricals
X = pd.get_dummies(X)

# Scale numeric features (important for stability)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# === Step 5: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"🧪 Training samples: {X_train.shape[0]}")
print(f"🧪 Testing samples:  {X_test.shape[0]}")

# === Step 6: Train XGBoost Model ===
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# === Step 7: Evaluate ===
preds = model.predict(X_test)

report = classification_report(y_test, preds, digits=4)
conf_matrix = confusion_matrix(y_test, preds)

print("📊 Classification Report:\n", report)
print("🧩 Confusion Matrix:\n", conf_matrix)

# === Step 8: Save Model & Feature Metadata ===
joblib.dump(model, os.path.join(MODEL_DIR, MODEL_NAME))
X.columns.to_series().to_json(
    os.path.join(MODEL_DIR, FEATURES_NAME),
    indent=2
)

print(f"✅ Model saved to: {MODEL_DIR}{MODEL_NAME}")
print(f"🧠 Feature list saved to: {MODEL_DIR}{FEATURES_NAME}")
print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Loaded dataset with shape: (7043, 21)
🧪 Training samples: 5625
🧪 Testing samples:  1407
📊 Classification Report:
               precision    recall  f1-score   support

           0     0.8369    0.8742    0.8551      1033
           1     0.6037    0.5294    0.5641       374

    accuracy                         0.7825      1407
   macro avg     0.7203    0.7018    0.7096      1407
weighted avg     0.7749    0.7825    0.7778      1407

🧩 Confusion Matrix:
 [[903 130]
 [176 198]]
✅ Model saved to: ../backend/models/telco/xgb_model.pkl
🧠 Feature list saved to: ../backend/models/telco/feature_names_xgb.json
🏁 Done at 2026-01-24 11:22:10
